In [5]:
import pandas as pd
import string
import re
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score

/home/gridsan/manderson/.conda/envs/skyscraper2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def normalize(x):
    x = str(x).strip().lower()
    x = x.translate(str.maketrans("", "", string.punctuation))
    x = re.sub(r"\s+", " ", x)
    return x

In [7]:
# Event detection

csv_path = "/home/gridsan/manderson/skyscraper-s2/out/vqa/detection_videollava.csv"
df = pd.read_csv(csv_path)

y_true = df["ground_truth"] == 'yes'
y_pred = df["prediction"] == 'yes'

print("accuracy:", accuracy_score(y_true, y_pred))
print("precision:", precision_score(y_true, y_pred, zero_division=0))
print("recall:", recall_score(y_true, y_pred, zero_division=0))
print("f1:", f1_score(y_true, y_pred, zero_division=0))

accuracy: 0.49066467513069456
precision: 0.7201946472019465
recall: 0.3429895712630359
f1: 0.46467817896389324


In [8]:
# Event classification

csv_path = "/home/gridsan/manderson/skyscraper-s2/out/vqa/classification_videollava.csv"
df = pd.read_csv(csv_path)

y_true = df["ground_truth"].apply(normalize)
y_pred = df["prediction"].apply(normalize)

print("accuracy:", accuracy_score(y_true, y_pred))
print("precision:", precision_score(y_true, y_pred, average="weighted", zero_division=0))
print("recall:", recall_score(y_true, y_pred, average="weighted", zero_division=0))
print("f1:", f1_score(y_true, y_pred, average="weighted", zero_division=0))

#print(classification_report(y_true, y_pred, zero_division=0))

accuracy: 0.34652725914861837
precision: 0.4110787177990217
recall: 0.34652725914861837
f1: 0.2639873950304145


In [11]:
# Event description

csv_path = "/home/gridsan/manderson/skyscraper-s2/out/vqa/description_videollava.csv"
df = pd.read_csv(csv_path)

# --- CIDEr ---
refs = {}
hyps = {}

for i, row in df.iterrows():
    refs[i] = [{"caption": str(row["ground_truth"])}]
    hyps[i] = [{"caption": str(row["prediction"])}]

tokenizer = PTBTokenizer()
refs_tok = tokenizer.tokenize(refs)
hyps_tok = tokenizer.tokenize(hyps)

cider_score, _ = Cider().compute_score(refs_tok, hyps_tok)
print("CIDEr:", cider_score)
print()

# --- BLEU ---
smooth = SmoothingFunction().method1

bleu1, bleu2, bleu3, bleu4 = [], [], [], []

for _, row in df.iterrows():
    ref = [str(row["ground_truth"]).split()]
    hyp = str(row["prediction"]).split()

    bleu1.append(sentence_bleu(ref, hyp, weights=(1,0,0,0), smoothing_function=smooth))
    bleu2.append(sentence_bleu(ref, hyp, weights=(0.5,0.5,0,0), smoothing_function=smooth))
    bleu3.append(sentence_bleu(ref, hyp, weights=(1/3,1/3,1/3,0), smoothing_function=smooth))
    bleu4.append(sentence_bleu(ref, hyp, weights=(0.25,0.25,0.25,0.25), smoothing_function=smooth))

print("BLEU-1:", sum(bleu1)/len(bleu1))
print("BLEU-2:", sum(bleu2)/len(bleu2))
print("BLEU-3:", sum(bleu3)/len(bleu3))
print("BLEU-4:", sum(bleu4)/len(bleu4))
print()

# --- ROUGE ---
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

r1, r2, rL = [], [], []

for _, row in df.iterrows():
    scores = scorer.score(str(row["ground_truth"]), str(row["prediction"]))
    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rL.append(scores["rougeL"].fmeasure)

print("ROUGE-1:", sum(r1)/len(r1))
print("ROUGE-2:", sum(r2)/len(r2))
print("ROUGE-L:", sum(rL)/len(rL))
print()

# --- BERTScore ---
preds = df["prediction"].astype(str).tolist()
gts = df["ground_truth"].astype(str).tolist()

P, R, F1 = score(preds, gts, lang="en", verbose=True)

print("BERTScore Precision:", P.mean().item())
print("BERTScore Recall:", R.mean().item())
print("BERTScore F1:", F1.mean().item())

PTBTokenizer tokenized 34197 tokens at 410520.98 tokens per second.
PTBTokenizer tokenized 27933 tokens at 379225.67 tokens per second.


CIDEr: 0.045924556412555086

BLEU-1: 0.20362641179292904
BLEU-2: 0.06333795384132694
BLEU-3: 0.024841712749097244
BLEU-4: 0.015026367743265729

ROUGE-1: 0.25554983779092433
ROUGE-2: 0.03181851315836761
ROUGE-L: 0.1741417736846092



Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2499.40it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


100%|██████████| 27/27 [00:09<00:00,  2.76it/s]


computing greedy matching.


100%|██████████| 14/14 [00:01<00:00,  9.68it/s]

done in 11.24 seconds, 78.27 sentences/sec
BERTScore Precision: 0.8624773621559143
BERTScore Recall: 0.8573222756385803
BERTScore F1: 0.8598565459251404
